In [3]:
import numpy as np
import pandas as pd

###############################################################################
# 0.1 - Cleaning and harmonizing the dyadic panel
###############################################################################

# Import the dataset (R: fread(..., sep=";"))
df = pd.read_csv("panel_network_unbalanced_all_group_3.csv", sep=";", low_memory=False)

# Drop all columns whose names start with "feature"
feature_cols = [c for c in df.columns if c.startswith("feature")]
df = df.drop(columns=feature_cols, errors="ignore")

# Missingness snapshot (top 5)
miss = df.isna().mean().sort_values(ascending=False)
miss_long = (
    miss.reset_index()
        .rename(columns={"index": "var", 0: "missing_rate"})
)
print(miss_long.head(5))

# id_dyad : order-invariant dyad identifier (i,j) and (j,i) share the same id
# (cast to numeric if possible; otherwise keep as-is)
id1 = pd.to_numeric(df["id_1"], errors="coerce")
id2 = pd.to_numeric(df["id_2"], errors="coerce")

# If ids are non-numeric strings, pmin/pmax equivalent still works lexicographically
a = df["id_1"].astype(str)
b = df["id_2"].astype(str)

use_numeric = id1.notna() & id2.notna()
min_id = np.where(use_numeric, np.minimum(id1, id2).astype("Int64").astype(str), np.minimum(a, b))
max_id = np.where(use_numeric, np.maximum(id1, id2).astype("Int64").astype(str), np.maximum(a, b))
df["id_dyad"] = (pd.Series(min_id, index=df.index) + "_" + pd.Series(max_id, index=df.index))

# agemean : mean age within the dyad
df["agemean"] = (pd.to_numeric(df["age_1"], errors="coerce") + pd.to_numeric(df["age_2"], errors="coerce")) / 2

# same_group : 1 if same group, 0 otherwise
# NOTE: your R code has typos "goup_id_1/2" -> using that exact spelling if present, else fallback
g1 = "goup_id_1" if "goup_id_1" in df.columns else "group_id_1"
g2 = "goup_id_2" if "goup_id_2" in df.columns else "group_id_2"
df["same_group"] = (df[g1] == df[g2]).astype("int64")

# Rename variables (match your R setnames)
rename_map = {
    "collab": "yearly_collabs_dyad",
    "link": "link_dyad",
    "goup_id_1": "group_id_1",
    "goup_id_2": "group_id_2",
    "type_1_1": "core_1",
    "type_2_1": "female_1",
    "type_1_2": "core_2",
    "type_2_2": "female_2",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

# Reorder columns: keep final_order first (only those that exist), then the rest
final_order = [
    "year","post","id_1","id_2","id_dyad","id_dyad_in_group",
    "group_id_1","group_id_2","group_treated_1","group_treated_2","same_group",
    "yearly_collabs_dyad","link_dyad",
    "core_1","core_2","female_1","female_2",
    "age_1","age_2","agemax","agedif","agemean",
    "outcome_1_1","outcome_1_2","outcome_2_1","outcome_2_2","outcome_3_1","outcome_3_2",
    "selection_1","selection_2",
    "field_1_1","field_2_1","field_3_1","field_4_1","field_5_1","field_6_1","field_7_1","field_8_1",
    "field_1_2","field_2_2","field_3_2","field_4_2","field_5_2","field_6_2","field_7_2","field_8_2",
]
cols_first = [c for c in final_order if c in df.columns]
cols_rest = [c for c in df.columns if c not in cols_first]
df = df[cols_first + cols_rest]

         var  missing_rate
0   type_2_1      0.007646
1   type_2_2      0.005781
2  field_3_2      0.000000
3  field_3_1      0.000000
4  field_4_1      0.000000


In [3]:
df_work = df[
    df["group_treated_1"].notna() &
    df["group_treated_2"].notna() &
    (df["selection_1"] == 1) &
    (df["selection_2"] == 1)
].copy()


In [5]:
###############################################################################
# NetworkX: pre vs post stats + comparison table (Pre, Post, Δ, %Δ)
# + breakdown by CORE (core vs non-core) AND SEX (female vs male)
###############################################################################
import numpy as np
import pandas as pd
import networkx as nx

# ----------------------------
# Helpers
# ----------------------------
def _clean_str(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    return s if s else np.nan

def build_group_graph_nx(
    df: pd.DataFrame,
    group_id: str,
    period: str = "pre",
    year_cutoff: int = 2012,
    same_group_only: bool = True,
    link_or_positive_collab: bool = True,
    weight_col: str = "yearly_collabs_dyad",
):
    """Within-group weighted graph for group_id, aggregated over pre/post period."""
    assert period in {"pre", "post"}

    d = df.copy()
    d = d.dropna(subset=["group_id_1", "group_id_2", "id_1", "id_2", "year"])
    d["group_id_1"] = d["group_id_1"].astype(str)
    d["group_id_2"] = d["group_id_2"].astype(str)
    d["id_1"] = d["id_1"].map(_clean_str)
    d["id_2"] = d["id_2"].map(_clean_str)
    d = d.dropna(subset=["id_1", "id_2"])

    if same_group_only:
        d = d[d["group_id_1"].eq(d["group_id_2"])]

    d = d[d["group_id_1"].eq(str(group_id)) & d["group_id_2"].eq(str(group_id))]

    if period == "pre":
        d = d[d["year"].astype(int) < year_cutoff]
    else:
        d = d[d["year"].astype(int) >= year_cutoff]

    if link_or_positive_collab:
        d = d[(d["link_dyad"].fillna(0).astype(int) == 1) | (d[weight_col].fillna(0) > 0)]

    if d.empty:
        return None

    # order-invariant dyads
    a = np.minimum(d["id_1"].values, d["id_2"].values)
    b = np.maximum(d["id_1"].values, d["id_2"].values)
    d2 = d.assign(_a=a, _b=b)

    edges = (
        d2.groupby(["_a", "_b"], as_index=False)[weight_col]
        .sum()
        .rename(columns={"_a": "u", "_b": "v", weight_col: "w"})
    )
    edges = edges[edges["u"].ne(edges["v"]) & (edges["w"] > 0)]
    if edges.empty:
        return None

    # nodes attributes (core=1 if ever core)
    nodes_raw = pd.concat(
        [
            d[["id_1", "female_1", "core_1"]].rename(columns={"id_1": "id", "female_1": "female", "core_1": "core"}),
            d[["id_2", "female_2", "core_2"]].rename(columns={"id_2": "id", "female_2": "female", "core_2": "core"}),
        ],
        ignore_index=True,
    )
    nodes_raw["id"] = nodes_raw["id"].map(_clean_str)
    nodes_raw = nodes_raw.dropna(subset=["id"])

    def first_non_na(x):
        x = x.dropna()
        return x.iloc[0] if len(x) else np.nan

    nodes = (
        nodes_raw.groupby("id", as_index=False)
        .agg(
            female=("female", first_non_na),
            core=("core", lambda x: np.nanmax(pd.to_numeric(x, errors="coerce"))),
        )
    )
    nodes["female"] = pd.to_numeric(nodes["female"], errors="coerce").fillna(0).astype(int)
    nodes["core"] = (
        pd.to_numeric(nodes["core"], errors="coerce")
        .replace([-np.inf, np.inf], np.nan)
        .fillna(0)
        .astype(int)
    )

    G = nx.Graph()
    for _, r in nodes.iterrows():
        G.add_node(r["id"], female=int(r["female"]), core=int(r["core"]))

    for _, r in edges.iterrows():
        u, v, w = r["u"], r["v"], float(r["w"])
        if u == v:
            continue
        if G.has_edge(u, v):
            G[u][v]["w"] += w
        else:
            G.add_edge(u, v, w=w)

    # ensure endpoints exist
    for u, v in list(G.edges()):
        if u not in G.nodes:
            G.add_node(u, female=0, core=0)
        if v not in G.nodes:
            G.add_node(v, female=0, core=0)

    return G


def graph_stats_nx(G: nx.Graph) -> dict:
    """Stats used for comparison table (+ core split + sex split)."""
    if G is None or G.number_of_nodes() == 0:
        return {}

    n = G.number_of_nodes()
    m = G.number_of_edges()

    dens = nx.density(G)

    comps = [len(c) for c in nx.connected_components(G)]
    n_comp = len(comps)
    largest_comp_share = (max(comps) / n) if comps else np.nan

    # degree (binary)
    deg = dict(G.degree())
    deg_vals = np.array(list(deg.values()), dtype=float)

    # strength (weighted degree) + total weight
    strength = {u: 0.0 for u in G.nodes()}
    total_w = 0.0
    for u, v, data in G.edges(data=True):
        w = float(data.get("w", 1.0))
        total_w += w
        strength[u] += w
        strength[v] += w
    str_vals = np.array(list(strength.values()), dtype=float)

    clustering_mean = float(np.mean(list(nx.clustering(G).values()))) if n > 1 else np.nan

    if n >= 3:
        deg_max = deg_vals.max()
        deg_centralization = float(np.sum(deg_max - deg_vals) / ((n - 1) * (n - 2)))
    else:
        deg_centralization = np.nan

    # Core split
    core_flag = np.array([int(G.nodes[u].get("core", 0)) for u in G.nodes()], dtype=int) == 1
    mean_deg_core = float(deg_vals[core_flag].mean()) if core_flag.any() else np.nan
    mean_deg_noncore = float(deg_vals[~core_flag].mean()) if (~core_flag).any() else np.nan
    mean_str_core = float(str_vals[core_flag].mean()) if core_flag.any() else np.nan
    mean_str_noncore = float(str_vals[~core_flag].mean()) if (~core_flag).any() else np.nan
    core_strength_share = (
        float(str_vals[core_flag].sum() / str_vals.sum())
        if str_vals.sum() > 0 and core_flag.any()
        else np.nan
    )

    # Sex split (female vs male)
    female_flag = np.array([int(G.nodes[u].get("female", 0)) for u in G.nodes()], dtype=int) == 1
    mean_deg_female = float(deg_vals[female_flag].mean()) if female_flag.any() else np.nan
    mean_deg_male = float(deg_vals[~female_flag].mean()) if (~female_flag).any() else np.nan
    mean_str_female = float(str_vals[female_flag].mean()) if female_flag.any() else np.nan
    mean_str_male = float(str_vals[~female_flag].mean()) if (~female_flag).any() else np.nan
    female_strength_share = (
        float(str_vals[female_flag].sum() / str_vals.sum())
        if str_vals.sum() > 0 and female_flag.any()
        else np.nan
    )

    return {
        # Global
        "n_nodes": n,
        "n_edges": m,
        "density": dens,
        "n_components": n_comp,
        "largest_comp_share": largest_comp_share,
        "mean_degree": float(deg_vals.mean()) if n else np.nan,
        "max_degree": float(deg_vals.max()) if n else np.nan,
        "mean_strength": float(str_vals.mean()) if n else np.nan,
        "total_collab_weight": float(total_w),
        "mean_clustering": clustering_mean,
        "degree_centralization": deg_centralization,

        # Core split
        "mean_degree_core": mean_deg_core,
        "mean_degree_noncore": mean_deg_noncore,
        "mean_collab_per_core": mean_str_core,
        "mean_collab_per_noncore": mean_str_noncore,
        "core_strength_share": core_strength_share,

        # Sex split
        "mean_degree_female": mean_deg_female,
        "mean_degree_male": mean_deg_male,
        "mean_collab_per_female": mean_str_female,
        "mean_collab_per_male": mean_str_male,
        "female_strength_share": female_strength_share,
    }


# ----------------------------
# Comparison table (Pre, Post, Δ, %Δ)
# ----------------------------
def comparison_table_pre_post(df: pd.DataFrame, group_id: str, year_cutoff: int = 2012) -> pd.DataFrame:
    G_pre = build_group_graph_nx(df, group_id=group_id, period="pre", year_cutoff=year_cutoff)
    G_post = build_group_graph_nx(df, group_id=group_id, period="post", year_cutoff=year_cutoff)

    pre = pd.Series(graph_stats_nx(G_pre), name="Pre")
    post = pd.Series(graph_stats_nx(G_post), name="Post")

    all_idx = pre.index.union(post.index)
    pre = pre.reindex(all_idx)
    post = post.reindex(all_idx)

    delta = (post - pre).rename("Δ (Post-Pre)")
    pct = (100 * delta / pre.replace(0, np.nan)).rename("%Δ")

    out = pd.concat([pre, post, delta, pct], axis=1)

    metric_order = [
        # Global
        "n_nodes", "n_edges", "density",
        "n_components", "largest_comp_share",
        "mean_degree", "max_degree",
        "mean_strength", "total_collab_weight",
        "mean_clustering", "degree_centralization",

        # Core split
        "mean_degree_core", "mean_degree_noncore",
        "mean_collab_per_core", "mean_collab_per_noncore",
        "core_strength_share",

        # Sex split
        "mean_degree_female", "mean_degree_male",
        "mean_collab_per_female", "mean_collab_per_male",
        "female_strength_share",
    ]
    out = out.reindex([m for m in metric_order if m in out.index] + [m for m in out.index if m not in metric_order])

    out = out.reset_index().rename(columns={"index": "metric"})
    out.insert(0, "group_id", str(group_id))

    # rounding
    for c in ["Pre", "Post", "Δ (Post-Pre)", "%Δ"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["Pre"] = out["Pre"].round(4)
    out["Post"] = out["Post"].round(4)
    out["Δ (Post-Pre)"] = out["Δ (Post-Pre)"].round(4)
    out["%Δ"] = out["%Δ"].round(2)

    return out


df_work = df[
    df["group_treated_1"].notna()
    & df["group_treated_2"].notna()
    & (df["selection_1"] == 1)
    & (df["selection_2"] == 1)
].copy()

tab_i13 = comparison_table_pre_post(df_work, group_id="i13", year_cutoff=2012)
print(tab_i13.to_string(index=False))

# In Jupyter:
# from IPython.display import display
display(tab_i13)
###############################################################################

group_id                  metric      Pre      Post  Δ (Post-Pre)     %Δ
     i13                 n_nodes 143.0000  184.0000       41.0000  28.67
     i13                 n_edges 298.0000  534.0000      236.0000  79.19
     i13                 density   0.0294    0.0317        0.0024   8.06
     i13            n_components   1.0000    3.0000        2.0000 200.00
     i13      largest_comp_share   1.0000    0.9620       -0.0380  -3.80
     i13             mean_degree   4.1678    5.8043        1.6365  39.27
     i13              max_degree  15.0000   24.0000        9.0000  60.00
     i13           mean_strength  10.1538   15.6848        5.5309  54.47
     i13     total_collab_weight 726.0000 1443.0000      717.0000  98.76
     i13         mean_clustering   0.3792    0.4220        0.0428  11.29
     i13   degree_centralization   0.0774    0.1005        0.0232  29.93
     i13        mean_degree_core   6.1538    9.1923        3.0385  49.37
     i13     mean_degree_noncore   3.7265    5.2468

,group_id,metric,Pre,Post,Δ (Post-Pre),%Δ
0,i13,n_nodes,143.0000,184.0000,41.0000,28.67
1,i13,n_edges,298.0000,534.0000,236.0000,79.19
2,i13,density,0.0294,0.0317,0.0024,8.06
3,i13,n_components,1.0000,3.0000,2.0000,200.00
4,i13,largest_comp_share,1.0000,0.9620,-0.0380,-3.80
5,i13,mean_degree,4.1678,5.8043,1.6365,39.27
6,i13,max_degree,15.0000,24.0000,9.0000,60.00
7,i13,mean_strength,10.1538,15.6848,5.5309,54.47
8,i13,total_collab_weight,726.0000,1443.0000,717.0000,98.76
9,i13,mean_clustering,0.3792,0.4220,0.0428,11.29


### Network Descriptive Analysis – Group i13 (Pre vs Post)

- **n_nodes**: The number of nodes increased from 143 to 184 (+28.67%), indicating a substantial expansion of the group’s size after the policy.

- **n_edges**: The number of links rose from 298 to 534 (+79.19%), showing a strong intensification of connectivity within the network.

- **density**: Network density slightly increased (+8.06%), meaning that the probability of connections between members became marginally higher.

- **n_components**: The number of connected components increased from 1 to 3 (+200%), suggesting a slight fragmentation of the network structure.

- **largest_comp_share**: The share of nodes in the largest component decreased by 3.8%, indicating that a small portion of the network became disconnected from the main cluster.

- **mean_degree**: The average number of collaborators per member increased by 39.27%, reflecting a broad rise in individual connectivity.

- **max_degree**: The most connected individual increased their number of collaborators by 60%, pointing to stronger hub formation.

- **mean_strength**: The average weighted collaboration intensity rose by 54.47%, indicating more repeated or stronger collaborations per member.

- **total_collab_weight**: Total collaboration volume nearly doubled (+98.76%), highlighting a major increase in overall research activity.

- **mean_clustering**: The clustering coefficient increased by 11.29%, suggesting stronger local collaboration triangles.

- **degree_centralization**: Centralization rose by 29.93%, meaning the network became more concentrated around key actors.

- **mean_degree_core**: Core members increased their average degree by 49.37%, reinforcing their structural importance.

- **mean_degree_noncore**: Non-core members also increased connectivity (+40.80%), indicating that growth was widespread.

- **mean_collab_per_core**: Average collaborations per core member increased by 46.15%, showing intensified core activity.

- **mean_collab_per_noncore**: Non-core members experienced an even stronger rise (+64.23%), suggesting relative catching-up dynamics.

- **core_strength_share**: The share of total collaboration weight attributed to the core decreased by 26.47%, indicating a redistribution of influence toward non-core members.

- **mean_degree_female**: Female members increased their average degree by 42.66%, showing improved connectivity.

- **mean_degree_male**: Male members increased their average degree by 40.42%, reflecting similar expansion dynamics.

- **mean_collab_per_female**: Average collaborations per female member rose by 42.31%, indicating higher participation intensity.

- **mean_collab_per_male**: Male collaboration intensity increased by 61.30%, showing particularly strong post-period growth.

- **female_strength_share**: The share of total collaboration weight attributed to women increased by 14.17%, suggesting improved relative positioning of female members in the network.

In [7]:
tab_i13 = comparison_table_pre_post(df_work, group_id="i13", year_cutoff=2012)
tab_i80 = comparison_table_pre_post(df_work, group_id="i80", year_cutoff=2012)

tab_both = pd.concat([tab_i13, tab_i80], ignore_index=True)
print(tab_both.to_string(index=False))

group_id                  metric      Pre      Post  Δ (Post-Pre)     %Δ
     i13                 n_nodes 143.0000  184.0000       41.0000  28.67
     i13                 n_edges 298.0000  534.0000      236.0000  79.19
     i13                 density   0.0294    0.0317        0.0024   8.06
     i13            n_components   1.0000    3.0000        2.0000 200.00
     i13      largest_comp_share   1.0000    0.9620       -0.0380  -3.80
     i13             mean_degree   4.1678    5.8043        1.6365  39.27
     i13              max_degree  15.0000   24.0000        9.0000  60.00
     i13           mean_strength  10.1538   15.6848        5.5309  54.47
     i13     total_collab_weight 726.0000 1443.0000      717.0000  98.76
     i13         mean_clustering   0.3792    0.4220        0.0428  11.29
     i13   degree_centralization   0.0774    0.1005        0.0232  29.93
     i13        mean_degree_core   6.1538    9.1923        3.0385  49.37
     i13     mean_degree_noncore   3.7265    5.2468

### Network Descriptive Analysis – Group i80 (Pre vs Post)

- **n_nodes**: The number of members increased by 20.45%, indicating moderate network expansion.

- **n_edges**: The number of active links more than doubled (+106.91%), showing a very strong rise in new collaborations between members.

- **density**: Network density increased by 42.33%, meaning members became substantially more interconnected relative to network size.

- **n_components**: The number of connected components decreased from 5 to 2 (−60%), indicating a strong reduction in fragmentation.

- **largest_comp_share**: The share of nodes in the largest component rose to 97.17%, showing near-complete structural integration of the network.

- **mean_degree**: The average number of collaborators per member increased by 71.78%, reflecting a major broad-based connectivity boost.

- **max_degree**: The most connected individual increased their links by 80%, indicating stronger hub formation.

- **mean_strength**: Average collaboration intensity per member rose by 89.16%, suggesting both more partners and more repeated interactions.

- **total_collab_weight**: Total collaboration volume increased by 127.85%, indicating a dramatic rise in overall activity.

- **mean_clustering**: The clustering coefficient increased by 11.84%, showing stronger local triadic closure.

- **degree_centralization**: Centralization rose by 51.88%, meaning the network became more structurally concentrated around key actors.

- **mean_degree_core**: Core members increased their average number of partners by 82.57%, reinforcing their structural dominance.

- **mean_degree_noncore**: Non-core members also expanded their connectivity (+49%), but less strongly than the core.

- **mean_collab_per_core**: Average collaborations per core member more than doubled (+105.20%), indicating very strong intensification within the elite.

- **mean_collab_per_noncore**: Non-core collaboration intensity increased by 48.99%, suggesting participation growth but at a slower pace.

- **core_strength_share**: The core’s share of total collaboration weight slightly declined (−5.26%), indicating mild relative redistribution despite absolute core growth.

- **mean_degree_female**: Female connectivity increased by 49.65%, showing substantial expansion in network participation.

- **mean_degree_male**: Male connectivity increased even more (+87.63%), suggesting stronger male-driven link formation post-period.

- **mean_collab_per_female**: Female collaboration intensity rose by 74.53%, indicating deeper engagement.

- **mean_collab_per_male**: Male collaboration intensity nearly doubled (+96.91%), showing very strong activity growth.

- **female_strength_share**: The share of collaboration weight attributed to women declined by 10.64%, suggesting that men captured a relatively larger portion of post-period activity.

## Overall Conclusion on the Treatment Effect

Overall, the treatment is associated with a substantial expansion and intensification of collaborative activity across treated groups, as reflected by strong increases in the number of nodes, links, average degree, and total collaboration weight.

At the structural level, the effect appears heterogeneous across groups: while some networks exhibit mild fragmentation alongside diffusion toward non-core members (suggesting broader participation), others display strong integration, rising density, and increased centralization around key actors.

Importantly, although core members experience significant absolute growth in collaborations, their relative dominance (measured by core_strength_share) does not always increase and in some cases declines, indicating partial redistribution of collaborative intensity toward peripheral actors.

Gender dynamics also evolve: both male and female connectivity and collaboration intensity increase after treatment, but relative gains differ across groups, sometimes leading to shifts in the distribution of collaboration weight.

Taken together, the treatment effect does not merely increase activity mechanically; it reshapes the internal structure of networks by simultaneously expanding participation, altering concentration patterns, and modifying the distribution of collaborative influence.